# CNN Hyperparameter Tuning

Uses **Optuna** to search over augmentation and model hyperparameters.  
Objective: **maximize unseen-environment accuracy**.

All shared classes (`CNN`, `AudioDataset`, `AggressiveMultiVariationAugmentation`, etc.)  
are imported from `cnn_shared.py` — no duplication.

**Parameters tuned:**
| Parameter | What it controls |
|-----------|------------------|
| `snr_min` / `snr_max` | SNR range for noise augmentation |
| `time_shift_ratio` | TimeShifting max shift fraction |
| `freq_mask_param` | FrequencyMasking width (0 = off) |
| `time_mask_param` | TimeMasking width (0 = off) |
| `vol_reduction_prob` | Probability of volume-reduction augmentation |
| `dropout_conv` | Dropout2d rate in conv blocks |
| `dropout_fc` | Dropout rate in FC layers |

### Install Optuna (if needed)

In [ ]:
import importlib, subprocess, sys
if importlib.util.find_spec('optuna') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'optuna'])
print('optuna ready')

### Imports

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    repo_dir = '/content/keyboard_sound'
    if not os.path.exists(repo_dir):
        !git clone https://github.com/ayushma18/keyboard_sound {repo_dir}
    %cd {repo_dir}
    !pip install -r requirements_colab.txt

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import matplotlib.pyplot as plt
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from torch.utils.data import DataLoader
from torchvision.transforms import Compose, ToTensor
from torchaudio.transforms import FrequencyMasking, TimeMasking
from sklearn.model_selection import train_test_split

# ── Import everything shared from cnn_shared.py ───────────────────────────────
from cnn_shared import (
    NoiseLibrary, AggressiveMultiVariationAugmentation, TimeShifting,
    AudioDataset, TrainingDataset,
    CNN, ConvBlock, init_weights, get_device,
    calculate_rms, add_noise_snr, add_white_noise_snr, add_gaussian_noise_snr, reduce_volume_db,
)

print('All imports OK')

### Paths

In [ ]:
data_path        = 'Data/combined/AULA-NUM'
unseen_data_path = 'Data/combined/test-num'
noise_path       = 'model/Noises'
tuned_model_path = f"model/CNN-Tuned-{data_path.split('/')[-1]}.pkl"

### Device & mel transform

In [ ]:
device      = get_device()
sample_rate = 44100
to_mel      = torchaudio.transforms.MelSpectrogram(sample_rate, n_mels=64, hop_length=300, n_fft=2048, win_length=1024)
mel_to_np   = lambda s: s.clamp(min=1e-9).log2()[0, :, :].numpy()
base_tf     = Compose([to_mel, mel_to_np, ToTensor()])
print(f'Device: {device}')

### Load Noise Library

In [ ]:
noise_library = NoiseLibrary(noise_path)
print('Noise library ready')

### Load Data & Split

In [ ]:
dataset        = AudioDataset(data_path)
unseen_dataset = AudioDataset(unseen_data_path)
NUM_CLASSES    = dataset.num_classes()

targets = [d[1].item() for d in dataset.dataset]
train_idx, tmp_idx = train_test_split(range(len(dataset)), test_size=0.3, stratify=targets)
val_idx,  test_idx = train_test_split(tmp_idx, test_size=0.33, stratify=[targets[i] for i in tmp_idx])

init_train = torch.utils.data.Subset(dataset, train_idx)
init_val   = torch.utils.data.Subset(dataset, val_idx)
init_test  = torch.utils.data.Subset(dataset, test_idx)

print(f'Classes: {NUM_CLASSES}  |  Train: {len(init_train)}  Val: {len(init_val)}  Test: {len(init_test)}  Unseen: {len(unseen_dataset)}')

### Tuning Configuration

Edit `SEARCH_SPACE` to widen or narrow the ranges.

In [ ]:
TUNING_N_TRIALS   = 30   # total Optuna trials
TUNING_MAX_EPOCHS = 80   # epochs per trial (short run; best params used for full retrain)
TUNING_BATCH_SIZE = 32

SEARCH_SPACE = {
    'snr_min':             (3.0,  15.0),
    'snr_max':             (20.0, 35.0),
    'time_shift_ratio':    (0.1,  0.6),
    'freq_mask_param':     (0,    20),    # int; 0 = disabled
    'time_mask_param':     (0,    20),    # int; 0 = disabled
    'vol_reduction_prob':  (0.1,  0.6),
    'dropout_conv':        (0.1,  0.4),
    'dropout_fc':          (0.3,  0.6),
}

print('Search space:')
for k, v in SEARCH_SPACE.items():
    print(f'  {k:25s}: {v}')

### Optuna Objective

In [ ]:
def build_aug_transforms(trial):
    snr_min            = trial.suggest_float('snr_min',           *SEARCH_SPACE['snr_min'])
    snr_max            = trial.suggest_float('snr_max',           *SEARCH_SPACE['snr_max'])
    time_shift_ratio   = trial.suggest_float('time_shift_ratio',  *SEARCH_SPACE['time_shift_ratio'])
    freq_mask_param    = trial.suggest_int(  'freq_mask_param',   *SEARCH_SPACE['freq_mask_param'])
    time_mask_param    = trial.suggest_int(  'time_mask_param',   *SEARCH_SPACE['time_mask_param'])
    vol_reduction_prob = trial.suggest_float('vol_reduction_prob',*SEARCH_SPACE['vol_reduction_prob'])

    aug = AggressiveMultiVariationAugmentation(
        noise_library=noise_library,
        snr_range=(snr_min, snr_max),
        volume_reduction_prob=vol_reduction_prob,
        volume_reduction_range=(2, 15),
        multi_aug_prob=0.3,
    )
    steps = [aug, TimeShifting(time_shift_ratio), to_mel, mel_to_np, ToTensor()]
    if freq_mask_param > 0:
        steps.append(FrequencyMasking(freq_mask_param))
    if time_mask_param > 0:
        steps.append(TimeMasking(time_mask_param))
    return Compose(steps)


def objective(trial):
    dropout_conv = trial.suggest_float('dropout_conv', *SEARCH_SPACE['dropout_conv'])
    dropout_fc   = trial.suggest_float('dropout_fc',   *SEARCH_SPACE['dropout_fc'])
    aug_tf       = build_aug_transforms(trial)

    # num_workers=0 is required inside Jupyter to avoid DataLoader deadlocks
    dl_kw = dict(batch_size=TUNING_BATCH_SIZE, num_workers=0)
    train_dl  = DataLoader(TrainingDataset(init_train,      aug_tf),  shuffle=True,  **dl_kw)
    val_dl    = DataLoader(TrainingDataset(init_val,        base_tf), shuffle=False, **dl_kw)
    unseen_dl = DataLoader(TrainingDataset(unseen_dataset,  base_tf), shuffle=False, **dl_kw)

    model = CNN(num_classes=NUM_CLASSES, dropout_conv=dropout_conv, dropout_fc=dropout_fc)
    model.apply(init_weights)
    model.to(device)

    opt       = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched     = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=10)
    criterion = nn.CrossEntropyLoss()

    best_unseen_acc = 0.0

    for epoch in range(TUNING_MAX_EPOCHS):
        model.train()
        epoch_loss = 0.0
        for inputs, labels in train_dl:
            inputs, labels = inputs.to(device), torch.squeeze(labels).to(device)
            opt.zero_grad()
            loss = criterion(model(inputs), labels)
            loss.backward()
            opt.step()
            epoch_loss += loss.item()

        model.eval()
        with torch.no_grad():
            val_loss = 0.0
            for inputs, labels in val_dl:
                inputs, labels = inputs.to(device), torch.squeeze(labels).to(device)
                val_loss += criterion(model(inputs), labels).item()
            sched.step(val_loss / len(val_dl))

            correct = total = 0
            for inputs, labels in unseen_dl:
                inputs, labels = inputs.to(device), torch.squeeze(labels).to(device)
                _, pred = torch.max(model(inputs), 1)
                correct += (pred == labels).sum().item()
                total   += labels.size(0)

        unseen_acc      = correct / total
        best_unseen_acc = max(best_unseen_acc, unseen_acc)

        trial.report(unseen_acc, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_unseen_acc

### Run Optuna Study

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=20),
)

print(f'Starting Optuna search: {TUNING_N_TRIALS} trials x {TUNING_MAX_EPOCHS} epochs each')
print(f'Objective: maximize unseen-environment accuracy\n')

study.optimize(objective, n_trials=TUNING_N_TRIALS, show_progress_bar=True)

print('\n' + '='*60)
print('TUNING COMPLETE')
print('='*60)
print(f'Best unseen accuracy : {study.best_value:.4f}')
print('Best hyperparameters :')
for k, v in study.best_params.items():
    print(f'  {k:25s} = {v}')

### Trial History Plot

In [ ]:
completed   = [t for t in study.trials if t.value is not None]
values      = [t.value for t in completed]
best_so_far = [max(values[:i+1]) for i in range(len(values))]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.scatter(range(len(values)), values, s=20, alpha=0.7, label='Trial unseen acc')
plt.plot(best_so_far, color='red', label='Best so far')
plt.xlabel('Trial'); plt.ylabel('Unseen accuracy')
plt.title('Optuna trial history'); plt.legend()

try:
    importances = optuna.importance.get_param_importances(study)
    plt.subplot(1, 2, 2)
    names = list(importances.keys())
    imps  = list(importances.values())
    plt.barh(names[::-1], imps[::-1])
    plt.xlabel('Importance'); plt.title('Hyperparameter importance')
except Exception:
    pass

plt.tight_layout(); plt.show()

### Full Retrain with Best Hyperparameters

In [ ]:
bp = study.best_params

FULL_MAX_EPOCHS          = 1200
FULL_EARLY_STOP_PATIENCE = 50
FULL_EARLY_STOP_DELTA    = 0.001

best_aug = AggressiveMultiVariationAugmentation(
    noise_library=noise_library,
    snr_range=(bp['snr_min'], bp['snr_max']),
    volume_reduction_prob=bp['vol_reduction_prob'],
    volume_reduction_range=(2, 15),
    multi_aug_prob=0.3,
)
steps = [best_aug, TimeShifting(bp['time_shift_ratio']), to_mel, mel_to_np, ToTensor()]
if bp['freq_mask_param'] > 0:
    steps.append(FrequencyMasking(bp['freq_mask_param']))
if bp['time_mask_param'] > 0:
    steps.append(TimeMasking(bp['time_mask_param']))
best_aug_tf = Compose(steps)

dl_kw     = dict(batch_size=32, num_workers=0)
train_dl  = DataLoader(TrainingDataset(init_train,     best_aug_tf), shuffle=True,  **dl_kw)
val_dl    = DataLoader(TrainingDataset(init_val,       base_tf),     shuffle=False, **dl_kw)
test_dl   = DataLoader(TrainingDataset(init_test,      base_tf),     shuffle=False, **dl_kw)
unseen_dl = DataLoader(TrainingDataset(unseen_dataset, base_tf),     shuffle=False, **dl_kw)

model     = CNN(num_classes=NUM_CLASSES, dropout_conv=bp['dropout_conv'], dropout_fc=bp['dropout_fc'])
model.apply(init_weights)
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
criterion = nn.CrossEntropyLoss()

print(f'Full retrain — up to {FULL_MAX_EPOCHS} epochs with best params')
for k, v in bp.items():
    print(f'  {k:25s} = {v}')

In [ ]:
train_losses, train_accs = [], []
val_losses,   val_accs   = [], []
unseen_accs  = []
best_val_acc = 0.0
no_improve   = 0

for epoch in range(FULL_MAX_EPOCHS):
    model.train()
    epoch_loss = correct = total = 0
    for inputs, labels in train_dl:
        inputs, labels = inputs.to(device), torch.squeeze(labels).to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        _, pred  = torch.max(outputs, 1)
        correct += (pred == labels).sum().item()
        total   += labels.size(0)
        epoch_loss += loss.item()

    train_losses.append(epoch_loss / len(train_dl))
    train_accs.append(correct / total)

    model.eval()
    with torch.no_grad():
        val_loss = val_c = val_t = 0
        for inputs, labels in val_dl:
            inputs, labels = inputs.to(device), torch.squeeze(labels).to(device)
            outputs  = model(inputs)
            val_loss += criterion(outputs, labels).item()
            _, pred  = torch.max(outputs, 1)
            val_c   += (pred == labels).sum().item()
            val_t   += labels.size(0)

        avg_val_loss = val_loss / len(val_dl)
        val_acc      = val_c / val_t
        val_losses.append(avg_val_loss)
        val_accs.append(val_acc)

        u_c = u_t = 0
        for inputs, labels in unseen_dl:
            inputs, labels = inputs.to(device), torch.squeeze(labels).to(device)
            _, pred = torch.max(model(inputs), 1)
            u_c += (pred == labels).sum().item()
            u_t += labels.size(0)
        unseen_acc = u_c / u_t
        unseen_accs.append(unseen_acc)

    scheduler.step(avg_val_loss)

    print(f'Epoch [{epoch+1}/{FULL_MAX_EPOCHS}]  '
          f'Train {train_accs[-1]:.4f}  Val {val_acc:.4f}  Unseen {unseen_acc:.4f}')

    if val_acc > best_val_acc + FULL_EARLY_STOP_DELTA:
        best_val_acc = val_acc
        no_improve   = 0
        torch.save(model.state_dict(), tuned_model_path)
        print(f'  Saved best model (val acc {val_acc:.4f})')
    else:
        no_improve += 1

    if no_improve >= FULL_EARLY_STOP_PATIENCE:
        print(f'\nEarly stopping at epoch {epoch+1}')
        break

    if (epoch + 1) % 20 == 0:
        plt.figure(figsize=(12, 4))
        plt.subplot(1,2,1); plt.title('Loss')
        plt.plot(train_losses, label='train'); plt.plot(val_losses, label='val'); plt.legend()
        plt.subplot(1,2,2); plt.title('Accuracy')
        plt.plot(train_accs, label='train'); plt.plot(val_accs, label='val')
        plt.plot(unseen_accs, '--', label='unseen'); plt.legend()
        plt.tight_layout(); plt.show()

print(f'\nDone.  Best val acc: {best_val_acc:.4f}  |  Final unseen acc: {unseen_accs[-1]:.4f}')
print(f'Model saved to: {tuned_model_path}')

### Final Results

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1,2,1); plt.title('Loss (full retrain)')
plt.plot(train_losses, label='train'); plt.plot(val_losses, label='val'); plt.legend()
plt.subplot(1,2,2); plt.title('Accuracy (full retrain)')
plt.plot(train_accs, label='train'); plt.plot(val_accs, label='val')
plt.plot(unseen_accs, '--', color='orange', label='unseen'); plt.legend()
plt.tight_layout(); plt.show()

print(f'Best val acc    : {max(val_accs):.4f}')
print(f'Best unseen acc : {max(unseen_accs):.4f}')
print(f'Final unseen acc: {unseen_accs[-1]:.4f}')
print(f'Epochs trained  : {len(val_accs)}')